# HW 1 — Applied: Seam Carving
**Modern Computer Vision** — Technion, Spring 2026

**Due:** April 26, 2026

**Student ID:**

In [ ]:
STUDENT_ID = ""  # <-- Fill in your student ID

---
## Background

**Seam carving** is a content-aware image resizing technique introduced by Avidan & Shamir (2007).
Instead of uniformly scaling an image (which distorts content) or cropping (which removes content),
seam carving removes the pixels that matter least — one connected seam at a time.

A **seam** is a connected path of pixels from top to bottom (vertical seam) or left to right (horizontal seam),
with exactly one pixel per row (or column). Adjacent pixels in the seam must be neighbors
(directly below, or diagonally below-left / below-right).

The algorithm:
1. Compute an **energy map** — how "important" each pixel is (we'll use gradient magnitude)
2. Use **dynamic programming** to find the seam with minimum total energy
3. **Remove** that seam from the image
4. Repeat until the image reaches the desired width

**Paper:** [Seam Carving for Content-Aware Image Resizing](https://faculty.idc.ac.il/arik/SCWeb/imret/imret.pdf) — Avidan & Shamir, SIGGRAPH 2007

In this assignment you will implement the full pipeline from scratch.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from PIL import Image
import numpy as np

%matplotlib inline

### Convolution helper

PyTorch's `F.conv2d` is designed for batched, multi-channel data (4D tensors) and computes
cross-correlation rather than convolution. Since we're working with plain 2D tensors,
here's a thin wrapper that handles the reshaping and kernel flip for you:

In [ ]:
def conv2d(image, kernel):
    """2D convolution on plain (H, W) tensors. Valid padding."""
    kernel_flipped = kernel.flip(0).flip(1)
    return F.conv2d(
        image.unsqueeze(0).unsqueeze(0),
        kernel_flipped.unsqueeze(0).unsqueeze(0)
    ).squeeze(0).squeeze(0)

---
## Load images

We provide two test images:
- **Couple portrait** — a photo with clear foreground subjects against a soft background, great for seeing how seam carving preserves important content
- **Table scene** — a wider composition with multiple objects, which we'll use later for 2D resizing

Both are bundled with this notebook — no download needed.

In [ ]:
def load_image(path):
    """Load an image as a float tensor in [0, 1], shape (H, W, 3)."""
    img = Image.open(path).convert('RGB')
    return torch.from_numpy(np.array(img)).float() / 255.0

def to_grayscale(img):
    """Convert (H, W, 3) RGB tensor to (H, W) grayscale."""
    return 0.2989 * img[:,:,0] + 0.5870 * img[:,:,1] + 0.1140 * img[:,:,2]

def show(img, title=""):
    """Display a tensor image. Works for (H,W) grayscale or (H,W,3) RGB."""
    plt.figure(figsize=(10, 6))
    if img.dim() == 2:
        plt.imshow(img.numpy(), cmap='gray')
    else:
        plt.imshow(img.numpy().clip(0, 1))
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
img = load_image("couple.jpg")
print(f"Image shape: {img.shape}")
show(img, "Couple portrait (original)")

---
## Part 1: Energy map

The energy of a pixel measures how much it stands out from its neighbors.
A simple and effective choice is the **gradient magnitude** — the sum of
absolute horizontal and vertical derivatives.

Use convolution with Sobel kernels to compute the gradients on the grayscale image:

$$
S_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}
\qquad
S_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}
$$

Then: $E(i,j) = |G_x(i,j)| + |G_y(i,j)|$

Note: the output of valid convolution is smaller than the input.
Pad the energy map back to the original size by replicating the border values.

In [ ]:
def compute_energy(gray):
    """
    Compute the energy map of a grayscale image.
    
    Args:
        gray: (H, W) tensor, grayscale image in [0, 1]
    Returns:
        (H, W) tensor, energy map (non-negative)
    """
    # YOUR CODE HERE
    # 1. Define Sobel kernels as torch tensors
    # 2. Convolve using the conv2d helper above
    # 3. Compute energy = |Gx| + |Gy|
    # 4. Pad back to original size
    #    (note: F.pad needs a 3D+ tensor for 2D padding, so unsqueeze first)
    #    e.g.: F.pad(e.unsqueeze(0), (1,1,1,1), mode='replicate').squeeze(0)
    raise NotImplementedError

In [ ]:
gray = to_grayscale(img)
energy = compute_energy(gray)

assert energy.shape == gray.shape, f"Energy shape {energy.shape} != image shape {gray.shape}"
assert (energy >= 0).all(), "Energy should be non-negative"

show(energy, "Energy map")
print(f"Energy range: [{energy.min():.4f}, {energy.max():.4f}]")

---
## Part 2: Find the minimum-energy vertical seam

A vertical seam is a path from the top row to the bottom row, choosing one pixel per row,
where consecutive pixels are vertically adjacent (straight down, down-left, or down-right).

Use **dynamic programming** to find the seam with the smallest total energy:

1. Initialize: $M[0, j] = E[0, j]$
2. Recurrence: $M[i, j] = E[i, j] + \min(M[i-1, j-1],\; M[i-1, j],\; M[i-1, j+1])$
3. The minimum value in the last row of $M$ is the seam's total cost
4. **Backtrack** from there to recover the seam path

Return the seam as a 1D tensor of column indices, one per row.

In [ ]:
def find_seam(energy):
    """
    Find the minimum-energy vertical seam.
    
    Args:
        energy: (H, W) tensor
    Returns:
        seam: (H,) integer tensor of column indices
    """
    # YOUR CODE HERE
    # 1. Build the cumulative cost matrix M (top to bottom)
    # 2. Find the minimum in the last row
    # 3. Backtrack to find the full seam path
    raise NotImplementedError

In [ ]:
seam = find_seam(energy)

assert seam.shape == (img.shape[0],), f"Seam shape {seam.shape}, expected ({img.shape[0]},)"
assert seam.dtype in (torch.int64, torch.int32, torch.long), "Seam should be integer indices"
assert (seam >= 0).all() and (seam < img.shape[1]).all(), "Seam indices out of bounds"

# Check connectivity: adjacent rows differ by at most 1 column
diffs = (seam[1:] - seam[:-1]).abs()
assert (diffs <= 1).all(), f"Seam is not connected! Max column jump: {diffs.max()}"

print(f"Seam found, total energy: {energy[torch.arange(len(seam)), seam].sum():.2f}")

In [ ]:
# Visualize the seam on the image
def show_seam(img, seam, title="", color=[1.0, 0.0, 0.0]):
    vis = img.clone()
    for i in range(len(seam)):
        vis[i, seam[i]] = torch.tensor(color)
    show(vis, title)

show_seam(img, seam, "Minimum energy seam")

---
## Part 3: Remove a seam

Given an image and a seam (column indices per row), produce a new image
that is one pixel narrower — the seam pixels have been removed.

This should work for both the RGB image `(H, W, 3)` and the grayscale `(H, W)`.

In [ ]:
def remove_seam(img, seam):
    """
    Remove a vertical seam from an image.
    
    Args:
        img:  (H, W) or (H, W, 3) tensor
        seam: (H,) integer tensor of column indices to remove
    Returns:
        (H, W-1) or (H, W-1, 3) tensor
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
img_removed = remove_seam(img, seam)
assert img_removed.shape == (img.shape[0], img.shape[1] - 1, 3), \
    f"Expected {(img.shape[0], img.shape[1]-1, 3)}, got {img_removed.shape}"

# Also test on grayscale
gray_removed = remove_seam(gray, seam)
assert gray_removed.shape == (gray.shape[0], gray.shape[1] - 1), \
    f"Expected {(gray.shape[0], gray.shape[1]-1)}, got {gray_removed.shape}"

print(f"Original: {img.shape[1]} → After removal: {img_removed.shape[1]}")

---
## Part 4: Full seam carving pipeline

Now put it all together. Write a function that removes `n_seams` vertical seams from an image.

In each iteration:
1. Convert current image to grayscale
2. Compute the energy map
3. Find the minimum seam
4. Remove the seam from both the RGB and grayscale images

In [ ]:
def seam_carve(img, n_seams, show_progress=True):
    """
    Remove n_seams vertical seams from img.
    
    Args:
        img:     (H, W, 3) RGB tensor
        n_seams: number of seams to remove
        show_progress: print progress every 10 seams
    Returns:
        (H, W - n_seams, 3) tensor
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Remove 200 seams for a dramatic result
n_remove = 200
result = seam_carve(img, n_remove)

assert result.shape == (img.shape[0], img.shape[1] - n_remove, 3)
print(f"\nOriginal size:    {img.shape[1]} x {img.shape[0]}")
print(f"After carving:    {result.shape[1]} x {result.shape[0]}")
print(f"Seams removed:    {n_remove}")

In [ ]:
# Compare: seam carving vs naive resize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img.numpy().clip(0,1))
axes[0].set_title(f"Original ({img.shape[1]}×{img.shape[0]})")
axes[0].axis('off')

axes[1].imshow(result.numpy().clip(0,1))
axes[1].set_title(f"Seam carving ({result.shape[1]}×{result.shape[0]})")
axes[1].axis('off')

# Naive resize for comparison
naive = F.interpolate(
    img.permute(2,0,1).unsqueeze(0),
    size=(img.shape[0], img.shape[1] - n_remove),
    mode='bilinear', align_corners=False
).squeeze(0).permute(1,2,0)
axes[2].imshow(naive.numpy().clip(0,1))
axes[2].set_title(f"Naive resize ({naive.shape[1]}×{naive.shape[0]})")
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Seam removal animation

Let's visualize how seam carving works step by step. The animation below highlights each seam
in red before removing it, so you can see the algorithm in action.

*Note: this uses your implementations above — if they work correctly, you'll see a smooth animation
of the image shrinking as low-energy seams are removed one by one.*

In [ ]:
def animate_seam_carving(img, n_seams, interval=150):
    """
    Create an animation showing seams being removed one at a time.
    Each frame highlights the next seam in red, then removes it.
    
    Args:
        img:      (H, W, 3) RGB tensor
        n_seams:  number of seams to remove and animate
        interval: milliseconds between frames
    Returns:
        HTML animation widget (plays in the notebook)
    """
    frames = []
    current = img.clone()
    
    for k in range(n_seams):
        gray_k = to_grayscale(current)
        energy_k = compute_energy(gray_k)
        seam_k = find_seam(energy_k)
        
        # Frame: current image with seam highlighted in red
        vis = current.clone()
        for i in range(current.shape[0]):
            vis[i, seam_k[i]] = torch.tensor([1.0, 0.0, 0.0])
        frames.append(vis.numpy().clip(0, 1))
        
        current = remove_seam(current, seam_k)
    
    # Final frame: the carved result
    frames.append(current.numpy().clip(0, 1))
    
    fig, ax = plt.subplots(figsize=(10, 6))
    # Pad all frames to same width (original) so animation doesn't resize
    max_w = frames[0].shape[1]
    padded = []
    for f in frames:
        h, w, c = f.shape
        pad = np.ones((h, max_w, c))  # white padding
        pad[:, :w, :] = f
        padded.append(pad)
    
    im = ax.imshow(padded[0])
    ax.axis('off')
    title = ax.set_title('')
    
    def update(frame_idx):
        im.set_data(padded[frame_idx])
        if frame_idx < n_seams:
            title.set_text(f'Removing seam {frame_idx + 1}/{n_seams}')
        else:
            title.set_text(f'Done! Removed {n_seams} seams')
        return [im, title]
    
    anim = FuncAnimation(fig, update, frames=len(padded), interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

# Animate 40 seams (enough to see the effect clearly; more takes longer to generate)
animate_seam_carving(img, 40, interval=120)

---
## Part 5: Seam insertion (image expansion)

So far we've removed seams to make images narrower. The opposite operation is **seam insertion**:
to widen an image, we can duplicate seams.

**Naive approach (greedy):** Find the lowest-cost seam, duplicate it, find the next one on the *modified* image, etc.
**Problem:** We'll keep duplicating the same seam, creating thick lines instead of distributing the duplication.

**Correct approach:**
1. Find *all* n seams on the **original** image by iteratively finding and removing seams
2. Track which seams were found at each step (remembering their original column positions)
3. Insert them back in reverse order (from right to left) so earlier insertions don't shift the later ones

This ensures each of the n lowest-energy seams is duplicated exactly once, spreading the expansion across the image.

### Implementation: `seam_insert(img, n_seams)`

The function should:
1. **Iteratively find seams** on the original image (like seam carving, but tracking seams instead of removing them)
2. **Map back to original coordinates** — keep track of where each seam was in the original image
3. **Insert pixels** — for each seam, insert a new pixel (average of its neighbors) at each position
4. **Insert in reverse order** (rightmost first) to avoid coordinate shifts

In [ ]:
def seam_insert(img, n_seams, show_progress=True):
    """
    Insert n_seams vertical seams into img (make it wider).
    
    Args:
        img:     (H, W, 3) RGB tensor
        n_seams: number of seams to insert
        show_progress: print progress
    Returns:
        (H, W + n_seams, 3) tensor
    """
    # YOUR CODE HERE
    # Strategy:
    # 1. Find n seams on the original image (iterate: compute energy, find seam, remove seam, record original position)
    # 2. Map the found seams back to original image coordinates
    # 3. Sort seams by position (to insert right-to-left)
    # 4. For each seam position, insert a new column (average of neighbors)
    # 5. Return the wider image
    raise NotImplementedError

In [ ]:
# Test seam insertion
n_insert = 100
result_insert = seam_insert(img, n_insert)

assert result_insert.shape == (img.shape[0], img.shape[1] + n_insert, 3), \
    f"Expected {(img.shape[0], img.shape[1] + n_insert, 3)}, got {result_insert.shape}"

print(f"Original size:       {img.shape[1]} × {img.shape[0]}")
print(f"After insertion:     {result_insert.shape[1]} × {result_insert.shape[0]}")
print(f"Seams inserted:      {n_insert}")

In [ ]:
# Compare: seam insertion vs naive upscale
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img.numpy().clip(0,1))
axes[0].set_title(f"Original ({img.shape[1]}×{img.shape[0]})")
axes[0].axis('off')

axes[1].imshow(result_insert.numpy().clip(0,1))
axes[1].set_title(f"Seam insertion ({result_insert.shape[1]}×{result_insert.shape[0]})")
axes[1].axis('off')

naive_up = F.interpolate(
    img.permute(2,0,1).unsqueeze(0),
    size=(img.shape[0], img.shape[1] + n_insert),
    mode='bilinear', align_corners=False
).squeeze(0).permute(1,2,0)
axes[2].imshow(naive_up.numpy().clip(0,1))
axes[2].set_title(f"Naive upscale ({naive_up.shape[1]}×{naive_up.shape[0]})")
axes[2].axis('off')

plt.tight_layout()
plt.show()

---
## Part 6: Two-dimensional content-aware resizing

Seam carving can resize in **both** dimensions. To make an image both narrower *and* shorter,
we remove vertical seams (reducing width) and horizontal seams (reducing height).

**Horizontal seam removal** works by transposing the image, running the same vertical seam carving,
and transposing back.

Let's try this on the table scene — a wider composition with distinct objects spread across it.

In [ ]:
# Load the table image
table_img = load_image("table_image.jpg")
print(f"Table image shape: {table_img.shape}")
show(table_img, "Table scene (original)")

In [ ]:
# 2D seam carving — alternate between vertical and horizontal seams
# so the image shrinks in both dimensions throughout.
n_vertical = 500
n_horizontal = 300
batch = 10  # seams per orientation before switching

current = table_img.clone()
v_done, h_done = 0, 0

while v_done < n_vertical or h_done < n_horizontal:
    # Batch of vertical seams (shrink width)
    v_todo = min(batch, n_vertical - v_done)
    if v_todo > 0:
        current = seam_carve(current, v_todo, show_progress=False)
        v_done += v_todo
    
    # Batch of horizontal seams (shrink height) via transpose
    h_todo = min(batch, n_horizontal - h_done)
    if h_todo > 0:
        current_t = current.permute(1, 0, 2)
        current_t = seam_carve(current_t, h_todo, show_progress=False)
        current = current_t.permute(1, 0, 2)
        h_done += h_todo
    
    print(f'\r  V: {v_done}/{n_vertical}  H: {h_done}/{n_horizontal}', end='')

table_2d = current
print(f'\nResult: {table_2d.shape[1]}×{table_2d.shape[0]}  (from {table_img.shape[1]}×{table_img.shape[0]})')

In [ ]:
# Compare at actual scale — pad smaller images onto the original canvas size
# so the size difference is immediately visible
H_orig, W_orig = table_img.shape[:2]
H_2d, W_2d = table_2d.shape[:2]

# Pad seam-carved result onto original-sized white canvas
canvas_carved = torch.ones(H_orig, W_orig, 3)
canvas_carved[:H_2d, :W_2d] = table_2d

# Naive resize to same target dimensions
naive_2d = F.interpolate(
    table_img.permute(2,0,1).unsqueeze(0),
    size=(H_2d, W_2d),
    mode='bilinear', align_corners=False
).squeeze(0).permute(1,2,0)
canvas_naive = torch.ones(H_orig, W_orig, 3)
canvas_naive[:H_2d, :W_2d] = naive_2d

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

axes[0].imshow(table_img.numpy().clip(0,1))
axes[0].set_title(f"Original ({W_orig}×{H_orig})", fontsize=13)
axes[0].axis('off')

axes[1].imshow(canvas_carved.numpy().clip(0,1))
axes[1].set_title(f"Seam carved ({W_2d}×{H_2d})", fontsize=13)
axes[1].axis('off')

axes[2].imshow(canvas_naive.numpy().clip(0,1))
axes[2].set_title(f"Naive resize ({W_2d}×{H_2d})", fontsize=13)
axes[2].axis('off')

plt.suptitle("2D Content-Aware Resizing — notice the size difference and how objects are preserved", fontsize=14)
plt.tight_layout()
plt.show()

### 2D seam carving animation

Watch the table image shrink in both dimensions — the animation alternates between
removing vertical seams (red lines top-to-bottom, width decreases) and horizontal seams
(red lines left-to-right, height decreases) in batches of 10. Every 5th seam is shown for speed.

In [ ]:
def animate_2d_seam_carving(img, n_vertical, n_horizontal, batch=10, step=10, interval=100):
    """
    Animate 2D seam carving with alternating vertical and horizontal seams.
    Removes `batch` seams of one orientation, then switches, so the image
    visibly shrinks in both dimensions throughout.
    
    Each captured frame highlights the seam in red (vertical = top-to-bottom,
    horizontal = left-to-right) before removing it.
    
    Args:
        img:          (H, W, 3) RGB tensor
        n_vertical:   total vertical seams to remove
        n_horizontal: total horizontal seams to remove
        batch:        seams per orientation before switching
        step:         capture a frame every `step` seams within each batch
        interval:     ms between animation frames
    """
    import matplotlib as mpl
    mpl.rcParams['animation.embed_limit'] = 50  # 50 MB limit
    
    frames = []
    H_orig, W_orig = img.shape[:2]
    current = img.clone()
    frames.append(('Original', current.numpy().clip(0,1)))
    
    v_done, h_done = 0, 0
    
    while v_done < n_vertical or h_done < n_horizontal:
        # --- Batch of vertical seams (top-to-bottom, shrinks width) ---
        v_todo = min(batch, n_vertical - v_done)
        for k in range(v_todo):
            gray_k = to_grayscale(current)
            energy_k = compute_energy(gray_k)
            seam_k = find_seam(energy_k)
            
            if (v_done + k + 1) % step == 0:
                vis = current.clone()
                for i in range(current.shape[0]):
                    vis[i, seam_k[i]] = torch.tensor([1.0, 0.0, 0.0])
                frames.append((f'V:{v_done+k+1}/{n_vertical}  H:{h_done}/{n_horizontal}',
                                vis.numpy().clip(0,1)))
            
            current = remove_seam(current, seam_k)
        v_done += v_todo
        
        # --- Batch of horizontal seams (left-to-right, shrinks height) ---
        h_todo = min(batch, n_horizontal - h_done)
        if h_todo > 0:
            current_t = current.permute(1, 0, 2)  # transpose: horizontal seams → vertical
            for k in range(h_todo):
                gray_k = to_grayscale(current_t)
                energy_k = compute_energy(gray_k)
                seam_k = find_seam(energy_k)
                
                if (h_done + k + 1) % step == 0:
                    vis_t = current_t.clone()
                    for i in range(current_t.shape[0]):
                        vis_t[i, seam_k[i]] = torch.tensor([1.0, 0.0, 0.0])
                    back = vis_t.permute(1, 0, 2)  # un-transpose: seam appears horizontal
                    frames.append((f'V:{v_done}/{n_vertical}  H:{h_done+k+1}/{n_horizontal}',
                                    back.numpy().clip(0,1)))
                
                current_t = remove_seam(current_t, seam_k)
            current = current_t.permute(1, 0, 2)
            h_done += h_todo
    
    frames.append((f'Done! {n_vertical}V + {n_horizontal}H removed', current.numpy().clip(0,1)))
    
    # Pad all frames to original size for consistent canvas
    padded = []
    for title, f in frames:
        h, w, c = f.shape
        pad = np.ones((H_orig, W_orig, c))
        pad[:h, :w, :] = f
        padded.append((title, pad))
    
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(padded[0][1])
    ax.axis('off')
    ttl = ax.set_title(padded[0][0], fontsize=13)
    
    def update(i):
        im.set_data(padded[i][1])
        ttl.set_text(padded[i][0])
        return [im, ttl]
    
    anim = FuncAnimation(fig, update, frames=len(padded), interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

# Animate: 500 vertical + 300 horizontal, alternating in batches of 10, showing every 10th seam
animate_2d_seam_carving(table_img, n_vertical, n_horizontal, batch=10, step=10, interval=80)

---
## Part 7: Try your own image

Apply seam carving to an image of your choice. Include the comparison plot (original vs seam carved vs naive resize).

Choose an image where seam carving makes a visible difference — images with a clear subject and a
relatively uniform background work well.

In [ ]:
# YOUR CODE HERE
# Load your own image and run seam carving on it.
# Show the comparison plot.


---
## Part 8: Questions

1. Where does the algorithm tend to remove seams? Why?
2. Can you think of an image where seam carving would fail badly? Why?
3. How does seam insertion compare to naive upscaling? When might it fail?

*Your answers here.*

---
*Modern Computer Vision — Technion — Spring 2026*